In [26]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yachaymarketing/text-coordinates-regions")

print("Path to dataset files:", path)

# Project paths (work whether the notebook is run from notebooks/ or the project root)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_SAMPLE = ROOT / "data" / "sample"
DATA_GOLD = ROOT / "data" / "gold"
ARTIFACTS = ROOT / "artifacts" / "embeddings"
RESULTS_PATH = ROOT / "results" / "metrics.csv"

SAMPLE_PATH = DATA_SAMPLE / "df_sample_100k.csv"
EMBEDDINGS_PATH = ARTIFACTS / "embeddings_100k.npy"
GOLD_ANNOTATED_PATH = DATA_GOLD / "gold_annotated.csv"
GOLD_SAMPLE_PATH = DATA_GOLD / "gold_sample.csv"
BOILERPLATE_CHECK_PATH = DATA_GOLD / "boilerplate_precision_check.csv"

Path to dataset files: /home/roman/.cache/kagglehub/datasets/yachaymarketing/text-coordinates-regions/versions/1


In [ ]:
import os
print(os.listdir(path))

['c_62.json', 'c_121.json', 'c_51.json', 'c_79.json', 'c_6.json', 'c_13.json', 'c_49.json', 'c_38.json', 'c_65.json', 'c_115.json', 'c_80.json', 'c_18.json', 'c_9.json', 'c_29.json', 'c_16.json', 'c_99.json', 'c_78.json', 'c_33.json', 'c_52.json', 'c_32.json', 'c_98.json', 'c_11.json', 'c_30.json', 'c_34.json', 'c_100.json', 'c_85.json', 'c_92.json', 'c_91.json', 'c_73.json', 'c_28.json', 'c_64.json', 'c_46.json', 'c_83.json', 'c_74.json', 'c_119.json', 'c_94.json', 'c_95.json', 'c_14.json', 'c_54.json', 'c_53.json', 'c_107.json', 'c_103.json', 'c_104.json', 'c_75.json', 'c_90.json', 'c_15.json', 'c_120.json', 'c_39.json', 'c_108.json', 'c_44.json', 'c_97.json', 'c_48.json', 'c_42.json', 'c_45.json', 'c_43.json', 'c_82.json', 'c_102.json', 'c_68.json', 'c_106.json', 'c_66.json', 'c_71.json', 'c_7.json', 'c_87.json', 'c_86.json', 'c_105.json', 'c_36.json', 'c_8.json', 'c_26.json', 'c_114.json', 'c_61.json', 'c_21.json', 'c_22.json', 'c_111.json', 'c_110.json', 'c_60.json', 'c_24.json', 

In [6]:
# Load all regional files into a single DataFrame, assigning a stable post_id
import json
import os
import pandas as pd

def load_region_file(filepath, region_id):
    """Read a JSONL file and tag every record with region and a stable post_id."""
    records = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            r = json.loads(line)
            r["region"] = region_id
            # post_id = region + original line number -> deterministic and unique
            r["post_id"] = f"{region_id}_{line_no}"
            records.append(r)
    return records

region_files = sorted(f for f in os.listdir(path) if f.endswith(".json"))
print(f"Total files: {len(region_files)}")

all_records = []
for fname in region_files:
    region_id = fname.replace("c_", "").replace(".json", "")
    all_records.extend(load_region_file(os.path.join(path, fname), region_id))

df = pd.DataFrame(all_records)
df["lon"] = df["coordinates"].apply(lambda c: float(c[0]))
df["lat"] = df["coordinates"].apply(lambda c: float(c[1]))

# keep the raw text untouched: boilerplate detection must run on it (with URLs)
df = df.rename(columns={"text": "raw_text"})

print("Total records:", len(df))
print("post_id unique:", df["post_id"].is_unique)
print("Columns:", df.columns.tolist())  # check for timestamp or other useful fields

Total files: 123
Total records: 615000
post_id unique: True
Columns: ['raw_text', 'coordinates', 'region', 'post_id', 'lon', 'lat']


In [8]:
# Text cleaning (operates on raw_text, keeps raw_text intact)
import re
import html

URL_RE = re.compile(r'https?://\S+')
MENTION_RE = re.compile(r'@\s*\w+')
SPACE_RE = re.compile(r'\s+')

def clean_text(text):
    text = html.unescape(text)
    text = URL_RE.sub('', text)        # URLs
    text = MENTION_RE.sub('', text)    # @username and "@ location"
    text = text.replace('#', '')       # drop hashtag symbol, keep the word
    return SPACE_RE.sub(' ', text).strip()

df["clean_text"] = df["raw_text"].apply(clean_text)

# sanity check: nothing should be left over
print("URL leftovers:", df["clean_text"].str.contains(r'https?://', regex=True).sum())
print("@ leftovers:", df["clean_text"].str.contains(r'@\s*\w+', regex=True).sum())

URL leftovers: 4
@ leftovers: 49


In [9]:
# Word count as a flag; the threshold is applied later (lets us test 3/4/5 as an ablation)
df["word_count"] = df["clean_text"].str.split().str.len()
df["is_too_short"] = df["word_count"] < 4

print(df["word_count"].describe())
print("Too short (<4 words):", df["is_too_short"].sum())

count    615000.000000
mean         10.570603
std          11.207545
min           0.000000
25%           3.000000
50%           7.000000
75%          14.000000
max          91.000000
Name: word_count, dtype: float64
Too short (<4 words): 189998


In [10]:
# Boilerplate detection on RAW text. Nothing is dropped: we only add a flag,
# so "with / without cleaning" can be tested later as an ablation.
# Note: t.co shortens every link, so domain-based detection is useless -> text patterns only.

boilerplate_text_patterns = [
    # Twitter system messages / meta templates
    r"account is temporarily unavailable because it violates",
    r"new landmark created! discover interesting places around",
    r"do you remember when you joined twitter",
    r"mytwitteranniversary",
    r"¿recuerdas cuando te uniste a",

    # "Just posted a photo/video" auto-templates, multiple languages
    r"just posted a (photo|video)",
    r"acaba de publicar (una|un) (foto|video)",         # Spanish
    r"acabou de publicar (uma|um) (foto|vídeo|video)",  # Portuguese
    r"acabei de publicar (uma|um) (foto|vídeo)",        # Portuguese (alt.)
    r"vient de publier (une|un) (photo|vidéo)",         # French
    r"только что опубликовано (фото|видео)",             # Russian
    r"опубликовал[а]? фото",
    r"mengirim (foto|video)",                            # Indonesian
    r"bir (fotoğraf|video) paylaştı",                    # Turkish (narrowed)
    r"foto appena pubblicata",                           # Italian
    r"hat gerade ein foto gepostet",                     # German
    r"har precis lagt upp ett foto",                     # Swedish
    r"μόλις δημοσίευσε μια φωτογραφί",                   # Greek

    # Foursquare/Swarm check-in: narrowed to "I'm at <place> ..." shape
    r"^i'?m at .{1,80}( \(|\s—|\s-\s|$)",

    # Time-remaining bot (Catalan)
    r"falten cinc minuts",
    r"passen cinc minuts",

    # Weather / earthquake automated bots
    r"usgs reports a",
    r"severe thunderstorm warning",
    r"cocorahs",                                   # CoCoRaHS precipitation reports
    r"co-op observer",                             # NWS co-op observer reports
    r"^at \d{1,2}:\d{2} [ap]m [a-z]{2,4}, .{0,60}\[[^\]]+\]",  # "at 7:00 am est, 3 w city [xx co, in] ..."

    # Booking/discovery bot
    r"discover (hotels|restaurants|attractions) around",
    r"discover things to do",

    # Referral / affiliate spam templates
    r"i'?m inviting you to join cashjunky",
    r"hey, i found this really awesome etsy",
    r"layanan yg bisa didapatkan di",

    # Coordinated religious-sect hashtag spam campaigns
    r"sant shri asharamji",
    r"bodhdiwas_of_saintrampalj",
    r"lifesavior_godkabir",
    r"constitutionofthesupremeg",
    r"mysteryofcreationofnature",
    r"रविदास_के_गुरु",
    r"बोधदिवस_संतरामप",
    r"الإمام ناصر محمد اليماني",
    r"imam nasser mohammad al-yemeni",
    r"request fridaylivestream",
]

text_regex = re.compile('|'.join(boilerplate_text_patterns), flags=re.IGNORECASE)

df["is_boilerplate"] = df["raw_text"].str.contains(text_regex, na=False)

print("Total boilerplate:", df["is_boilerplate"].sum(),
      f"({df['is_boilerplate'].mean():.1%})")

/tmp/ipykernel_30640/2327031064.py:66: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["is_boilerplate"] = df["raw_text"].str.contains(text_regex, na=False)


Total boilerplate: 15043 (2.4%)


In [11]:
# Precision check: save 200 random flagged posts for manual review
bp_sample = (df[df["is_boilerplate"]]
             .sample(n=200, random_state=42)
             [["post_id", "raw_text"]]
             .copy())
bp_sample["is_correct"] = ""   # fill in: 1 = real boilerplate, 0 = false positive
bp_sample.to_csv(BOILERPLATE_CHECK_PATH, index=False)

In [12]:
# Which domains appear most often in raw text? (to verify/extend the URL patterns)
domains = (df["raw_text"].str.extractall(r'https?://(?:www\.)?([^/\s]+)')[0]
           .str.lower().value_counts().head(30))
print(domains)

0
t.co                                   248365
instagram.com                              21
youtu.be                                   16
twitter.com                                 6
youtube.com                                 2
vm.tiktok.com                               2
fb.watch                                    2
facebook.com                                2
t.me                                        1
chat.whatsapp.com                           1
lucienwood20.picfair.com#cloudscape         1
t.                                          1
miladbodrum.com.tr                          1
nasser-moha                                 1
mahdialumma.                                1
t                                           1
tenies-online.best                          1
164.100.251.19                              1
wa.me                                       1
benefieldbookmark.com                       1
buymeacoffee,con                            1
abc.net.au                      

As we can see above: Twitter substituted all urls with t.co/path, so domain-based detection is useless. We cannot say, if this url is a boilerplate then, bc people really can may just genuinely share a link to a valid resource, compare:
 - just posted a photo @ tampa, florida https://t.co/3ylmsuzudd - boilerplate
 - this game was insane, watch the highlights https://t.co/abc123 - not boilerplate

In [13]:
# Exact-duplicate detection. Nothing is dropped here, only flagged.
# Normalized key: lowercase, no punctuation, collapsed spaces
# (so "Good morning!" and "good morning" count as the same text).
df["dup_key"] = (df["clean_text"].str.lower()
                 .str.replace(r"[^\w\s]", "", regex=True)
                 .str.replace(r"\s+", " ", regex=True)
                 .str.strip())

# Posts with no letters/digits at all (e.g. emoji-only) have an empty key.
# They carry no text signal, so they are excluded from the pool instead of being
# treated as duplicates of each other.
df["has_text"] = df["dup_key"].str.len() > 0

# Duplicates are searched only among posts that would otherwise be kept
base_ok = ~df["is_too_short"] & ~df["is_boilerplate"] & df["has_text"]

df["is_duplicate"] = False
# keep="first": the first occurrence stays, all repeats get the flag
df.loc[base_ok, "is_duplicate"] = df.loc[base_ok].duplicated("dup_key", keep="first")


print("Posts without text (emoji/punctuation only):", (~df["has_text"]).sum())
print("Eligible posts:", base_ok.sum())
print("Exact duplicates:", df["is_duplicate"].sum(),
      f"({df['is_duplicate'].sum() / base_ok.sum():.1%} of eligible)")
print(df.loc[base_ok, "dup_key"].value_counts().head(15))

Posts without text (emoji/punctuation only): 121887
Eligible posts: 372005
Exact duplicates: 10077 (2.7% of eligible)
dup_key
_             932
__            473
___           279
in            243
2             189
_ _           177
3             140
1             134
survivorgr    106
5             101
____           94
4              93
10             83
100            67
12             65
Name: count, dtype: int64


In [ ]:
# Recall check: most frequent 3-word prefixes among posts NOT flagged as boilerplate.
# Templates show up as unusually high counts for a single prefix.
unflagged = df.loc[~df["is_boilerplate"] & df["has_text"] & ~df["is_too_short"], "clean_text"].str.lower()
prefix3 = unflagged.str.split().str[:3].str.join(" ")
print(prefix3.value_counts().head(40))

clean_text
thank you for         241
thank you so          170
this is the           162
this is a             151
happy birthday to     149
happy new year        142
i want to             126
one of the            102
i need to              98
i have a               96
i love you             90
जय श्री राम            90
thanks for the         85
i feel like            82
i wish i               81
que ganas de           79
i love this            73
this is so             71
one of my              67
😂 😂 😂                  64
i just want            63
this is what           63
you are a              62
i need a               59
i think i              58
i am so                57
i think the            57
there is no            57
اللهم صل وسلم          55
how do you             53
we are not             53
i don’t know           52
e vamos de             52
जय जय श्री             51
the fact that          50
if you are             50
looking forward to     49
i don’t think          49
w

In [28]:
# Fixed 100k subsample used for ALL experiments.
# Pool = unique, non-trivial posts. Boilerplate stays IN the pool (only flagged),
# so the cleaning ablation can use exactly the same posts.
from sentence_transformers import SentenceTransformer
import time
import numpy as np

pool = df[~df["is_too_short"] & df["has_text"] & ~df["is_duplicate"]]
df_sample = pool.sample(n=100_000, random_state=42).reset_index(drop=True)
df_sample["is_clean"] = ~df_sample["is_boilerplate"]

print("Pool size:", len(pool))
print("Subsample size:", len(df_sample))
print("Boilerplate in sample:", (~df_sample["is_clean"]).sum())

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Embed ALL 100k posts (noisy ones included): embeddings do not depend on the flags
start = time.time()
embeddings = model.encode(
    df_sample["clean_text"].tolist(),
    show_progress_bar=True,
    batch_size=64
)
print(f"Done in {(time.time() - start) / 60:.1f} min, shape: {embeddings.shape}")

Pool size: 420952
Subsample size: 100000
Boilerplate in sample: 3489


Batches: 100%|██████████| 1563/1563 [20:20<00:00,  1.28it/s]


Done in 20.3 min, shape: (100000, 384)


In [ ]:
# Save embeddings and the corresponding sampled dataframe.
# Row order of the embeddings matches df_sample.
np.save(EMBEDDINGS_PATH, embeddings)
df_sample.to_csv(SAMPLE_PATH, index=False, encoding='utf-8')
print("Saved.")

Saved.


In [14]:
# Load previously saved embeddings and sample (if starting a new session).
# Explicit dtypes: otherwise pandas may re-type columns (e.g. region -> int)
# and turn strings like "NA" inside texts into NaN.
import numpy as np
import pandas as pd

embeddings = np.load(EMBEDDINGS_PATH)
df_sample = pd.read_csv(
    SAMPLE_PATH,
    dtype={"post_id": str, "region": str, "raw_text": str, "clean_text": str, "dup_key": str},
    keep_default_na=False,
)

assert len(df_sample) == embeddings.shape[0], (len(df_sample), embeddings.shape)

print("Embeddings shape:", embeddings.shape)
print("Sample size:", len(df_sample))
print(df_sample.dtypes)

Embeddings shape: (100000, 384)
Sample size: 100000
raw_text              str
coordinates           str
region                str
post_id               str
lon               float64
lat               float64
clean_text            str
word_count          int64
is_too_short         bool
bp_by_url            bool
bp_by_text           bool
is_boilerplate       bool
dup_key               str
is_duplicate         bool
has_text             bool
is_clean             bool
lang                  str
dup_count           int64
dtype: object


Added a column that tells how many duplicates each record has in original dataframe

In [15]:
counts = df.loc[base_ok].groupby("dup_key").size()
df_sample["dup_count"] = df_sample["dup_key"].map(counts).fillna(1).astype(int)

assert len(df_sample) == embeddings.shape[0]
print(df_sample["dup_count"].describe())
print(df_sample["dup_count"].value_counts().head())

df_sample.to_csv(SAMPLE_PATH, index=False, encoding='utf-8')

count    100000.000000
mean          1.007100
std           0.586032
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max         177.000000
Name: dup_count, dtype: float64
dup_count
1    99693
2      225
3       44
4       13
6        8
Name: count, dtype: int64


In [2]:
# Language detection with lingua (all 75 supported languages).
# Short informal posts -> results are approximate; used for language-mixing analysis of clusters.
from lingua import LanguageDetectorBuilder
import time

detector = (LanguageDetectorBuilder.from_all_languages()
            .with_preloaded_language_models()
            .build())

texts = df_sample["clean_text"].tolist()

start = time.time()
detected = detector.detect_languages_in_parallel_of(texts)   # list of Language or None
print(f"Done in {(time.time() - start) / 60:.1f} min")

# ISO 639-1 code in lowercase ("en", "es", ...); "und" = lingua could not decide (None)
df_sample["lang"] = [lang.iso_code_639_1.name.lower() if lang is not None else "und"
                     for lang in detected]

print("Undetermined:", (df_sample["lang"] == "und").sum())
print(df_sample["lang"].value_counts().head(20))
print("Languages found:", df_sample["lang"].nunique())

Done in 0.8 min
Undetermined: 114
lang
en    35384
es    15800
pt    12088
ar     4757
tr     3216
fr     2741
id     2371
tl     2101
hi     1985
ru     1594
it     1205
ms     1079
sw     1028
ja      824
el      715
de      698
th      677
sv      675
ko      630
ca      586
Name: count, dtype: int64
Languages found: 76


In [3]:
df_sample.to_csv(SAMPLE_PATH, index=False, encoding='utf-8')
print("Saved:", len(df_sample), "rows | columns:", df_sample.shape[1])

Saved: 100000 rows | columns: 17


## Near-duplicate check
Estimate how many posts have a near-identical neighbour in the sample (cosine similarity of MiniLM embeddings). This only measures the scale of the problem; nothing is removed yet.

In [16]:
# L2-normalise so that the dot product equals cosine similarity
X = (embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)).astype(np.float32)

nearest = np.zeros(len(X), dtype=np.float32)
chunk = 2000  # keeps the similarity block at ~800 MB
for i in range(0, len(X), chunk):
    sims = X[i:i + chunk] @ X.T
    sims[np.arange(sims.shape[0]), np.arange(i, i + sims.shape[0])] = -1  # ignore self-match
    nearest[i:i + chunk] = sims.max(axis=1)

for thr in (0.99, 0.95, 0.90):
    print(f"share of posts with a neighbour above {thr}: {(nearest > thr).mean():.3%}")

share of posts with a neighbour above 0.99: 1.601%
share of posts with a neighbour above 0.95: 4.009%
share of posts with a neighbour above 0.9: 8.953%


In [17]:
# Look at a few concrete pairs to judge whether "near-duplicate" really means duplicate
SEED = 42
rng = np.random.default_rng(SEED)
candidates = np.where((nearest > 0.95) & (nearest <= 0.99))[0]
sample_idx = rng.choice(candidates, size=15, replace=False)

for i in sample_idx:
    sims = X[i] @ X.T
    sims[i] = -1                      # exclude the post itself
    j = int(sims.argmax())            # index of its nearest neighbour
    print(f"sim={sims[j]:.3f}")
    print("  A:", df_sample.loc[i, "clean_text"][:120])
    print("  B:", df_sample.loc[j, "clean_text"][:120])

sim=0.950
  A: يارب تنطردي زيي ع تركيا😂
  B: لا وايه لازم تعرف انت منين 😂😂
sim=0.959
  A: cada commm a sua vibeee mirmaaa🥰🥰🙌🏾 obrigadaaaa
  B: intresiesst du dich wirklich für diesen typ🤨🙈😭😭😭
sim=0.982
  A: والله إنك صادق وإنهم آذونا بأنوار السفاتي .. ليتك باقي .. راحت
  B: حفظك الله أخي مصطفى..شكرآ جزيلآ🌸🍃
sim=0.958
  A: καλημέρα νικόλα μου ☕🙋
  B: buongiorno caro giuseppe 😘☕️
sim=0.965
  A: صبحكم الله بالخير ياحلوين ❤️
  B: اسعدالله صباحكم بكل سعاده وراحة بال☕️🌹
sim=0.977
  A: just posted a photo , ashanti, ghana
  B: just posted a photo , ghana
sim=0.976
  A: عندك شك😂🙂 اي صح
  B: te fâche pas waaahh😭😭
sim=0.957
  A: just posted a photo vegas, new mexico
  B: just posted a photo , mexico
sim=0.976
  A: وعلیکم السلام ورحمتہ اللہ وبرکاتہ ❤️ بیشک
  B: تبارك الرحمن،الله يخليكم لبعضو و يديم الود و الألفة✨
sim=0.966
  A: acabou de publicar uma foto em cachoeira dourada, goias, brazil
  B: acabou de publicar uma foto em são paulo, brazil
sim=0.976
  A: ستوري الانستا عبارة عن منشين ل احمد طه

## Do near-duplicates overlap with the boilerplate flag?
Among posts that have a very similar neighbour (cosine > 0.95), check how many are already flagged as boilerplate. If most of them are flagged, the flag already covers real template duplicates.

In [18]:
# Boolean boilerplate flag (adjust the column name if yours is different)
flag = df_sample["is_boilerplate"].astype(str).str.lower().eq("true").to_numpy()
high = nearest > 0.95

print(f"boilerplate share in the whole sample:             {flag.mean():.2%}")
print(f"boilerplate share among posts with neighbour>0.95: {flag[high].mean():.2%}")
print(f"posts with neighbour>0.95 and NOT flagged:         {(high & ~flag).mean():.2%} of the sample")

# Examples of unflagged posts that have a near-identical neighbour
unflagged_high = np.where(high & ~flag)[0]
rng = np.random.default_rng(SEED)
for i in rng.choice(unflagged_high, size=10, replace=False):
    print(round(float(nearest[i]), 3), "|", df_sample.loc[i, "clean_text"][:100])

boilerplate share in the whole sample:             3.49%
boilerplate share among posts with neighbour>0.95: 39.01%
posts with neighbour>0.95 and NOT flagged:         2.44% of the sample
0.998 | wondering what your gym is worth? the top 5 reasons to obtain a business valuation are: 1. planning 
0.994 | اللهم إنا في سترك لعيوبنا،وفي أمر عبادك بالستر علينا بشارةً بالمغفرة فما كنت لتستر في دار الفناء لتف
0.973 | uiui vc gostosa 😻
0.954 | الله يسلمك يارب ما تشوفي شر
0.968 | roah 041239z 10018g29kt 9999 -shra few009 bkn013 27/26 q0998 rmk 2cu009 6cu013 a2948
0.976 | وعلیکم السلام ورحمتہ اللہ وبرکاتہ ❤️ بیشک
0.982 | وأمّا بَدْر مساء يوم الأحد ليلة الإثنين فهو لَكُم فلا حاجة لي به. الإمام المهديّ ناصر محمد اليماني 0
0.976 | عندك شك😂🙂 اي صح
0.993 | حفظك الله أخي مصطفى..شكرآ جزيلآ🌸🍃
0.963 | 🎼🎶 ya vienen los reyes magos🎶 logroñoesnavidad felicesreyes logroño 🎄🎁🤴🤴🤴


## Near-duplicate check: conclusion
For every post in the 100k sample we found its most similar other post (cosine similarity of MiniLM embeddings). 4.0% of posts have a neighbour above 0.95 (1.6% above 0.99, 9.0% above 0.90).

Manual inspection of sampled pairs showed that a high similarity is a poor proxy for duplication in this data. Genuine duplicates are mostly templated texts that differ in a slot (a place name, a number, a date). The rest are short reactions, greetings and prayers that share a genre and emoji but not content. Cross-language greetings with similar meaning also score high, which is the desired behaviour of a multilingual encoder rather than a defect.

The boilerplate flag already covers 39% of the posts with a near-identical neighbour, although boilerplate makes up only 3.5% of the sample, so the filter catches the right kind of text. The remaining 2.4% of the sample contains some unflagged templates (METAR weather reports, religious broadcast messages, ad spam, hashtag campaigns), roughly 4 of 10 in a small manual sample, which suggests about 1% of the sample.

Decision: no near-duplicate filtering. The affected share is small, and removing by embedding similarity would also delete legitimate short posts. The unflagged templates are noted as a limitation of the regex filter.

## Gold set

Clustering is unsupervised, and silhouette cannot tell us whether clusters match real topics (it is inflated after UMAP and not comparable across representations). To evaluate methods fairly, we draw a random sample of 500 clean posts and label their topics (LLM annotation, validated manually on a subset). The sample is split into dev (model and parameter selection) and test (final report only), so tuning and reporting never use the same posts.

In [11]:
# Gold set sampling: random posts from the CLEAN part of the fixed 100k sample.
# Cluster labels are deliberately not used, to avoid selection bias.
GOLD_N = 3000
SEED = 42

clean_pool = df_sample[~df_sample["is_boilerplate"]]
gold = clean_pool.sample(n=GOLD_N, random_state=SEED)[["post_id", "clean_text"]].copy()

# dev (2/3): choosing parameters; test (1/3): touched only for the final report
rng = np.random.default_rng(SEED)
n_dev = int(GOLD_N * 2 / 3)
split = np.array(["test"] * GOLD_N, dtype=object)
split[rng.permutation(GOLD_N)[:n_dev]] = "dev"
gold["split"] = split

# empty columns to be filled during annotation
gold["topic"] = ""
gold["lang_true"] = ""

assert gold["post_id"].is_unique
print(gold["split"].value_counts())
print("Share of each language in the gold sample (lingua labels, for orientation only):")
print(df_sample.set_index("post_id").loc[gold["post_id"], "lang"].value_counts().head(10))

gold.to_csv(GOLD_SAMPLE_PATH, index=False, encoding="utf-8")
print(f"Saved {GOLD_SAMPLE_PATH.name}:", len(gold), "rows")

split
dev     2000
test    1000
Name: count, dtype: int64
Share of each language in the gold sample (lingua labels, for orientation only):
lang
en    1030
es     476
pt     335
ar     147
tr     115
fr      83
tl      75
id      70
hi      65
ru      41
Name: count, dtype: int64
Saved gold_sample.csv: 3000 rows


## Load data
Load the fixed 100k sample, the MiniLM embeddings and the gold annotations. The dev/test split is attached to gold from the original `gold_sample.csv`. Check that row order and post ids are consistent.

In [1]:
import numpy as np
import pandas as pd

# Fixed sample and embeddings (row order must match)
df_sample = pd.read_csv(
    SAMPLE_PATH,
    dtype={"post_id": str, "region": str, "raw_text": str, "clean_text": str, "dup_key": str},
    keep_default_na=False,
)
embeddings = np.load(EMBEDDINGS_PATH)
assert len(df_sample) == embeddings.shape[0], (len(df_sample), embeddings.shape)

# Gold annotations + dev/test split from the original gold sample
gold = pd.read_csv(GOLD_ANNOTATED_PATH, dtype=str, keep_default_na=False)
split = pd.read_csv(GOLD_SAMPLE_PATH, dtype=str, keep_default_na=False)[["post_id", "split"]]
gold = gold.merge(split, on="post_id", how="left", validate="one_to_one")

assert gold["post_id"].isin(df_sample["post_id"]).all(), "gold contains post_ids missing from the sample"
assert gold["split"].notna().all(), "some gold posts have no split"
assert (gold["topic"] != "").all(), "empty topic labels in gold"

print("sample:", df_sample.shape, "| embeddings:", embeddings.shape, "| gold:", gold.shape)
print(gold["split"].value_counts())
print(gold["topic"].value_counts())

sample: (100000, 18) | embeddings: (100000, 384) | gold: (3000, 5)
split
dev     2000
test    1000
Name: count, dtype: int64
topic
reaction_chat                   441
politics_news                   437
sports                          309
daily_life_work_study           254
family_friends_relationships    246
entertainment                   219
money_business                  167
religion                        130
health_covid                    125
travel_places_nature            120
tech_social_media               114
food_drink                      113
greetings_wishes                102
music                            97
unclear_other                    71
weather                          55
Name: count, dtype: int64


## Evaluation helper
Define `evaluate()`: compares cluster labels with the gold topics (ARI, NMI, homogeneity, completeness, V-measure) and measures language mixing inside clusters. Evaluated on two label sets: all gold labels, and topical labels only (without `reaction_chat` and `unclear_other`). Sanity-checked on synthetic labelings before any real clustering is run.

In [2]:
from scipy.stats import entropy
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_completeness_v_measure,
)

NON_TOPICAL = {"reaction_chat", "unclear_other"}

# we check if our model is capable of grouping by topic, regardless of language
def language_entropy(labels, langs):
    """Size-weighted mean normalized language entropy per cluster (0 = one language, 1 = uniform)."""
    d = pd.DataFrame({"cluster": labels, "lang": langs})
    d = d[(d["cluster"] != -1) & (d["lang"] != "und")]
    ents, weights = [], []
    for _, grp in d.groupby("cluster"):
        p = grp["lang"].value_counts(normalize=True).values
        ents.append(entropy(p) / np.log(max(len(p), 2)))
        weights.append(len(grp))
    return float(np.average(ents, weights=weights))


def evaluate(labels, df_sample, gold, split="dev", exclude_topics=()):
    """Evaluate a clustering of df_sample (labels are in df_sample row order) against gold.

    Noise points (label -1, e.g. from HDBSCAN) are excluded from the gold metrics;
    their share is reported separately as noise_share / gold_coverage.
    """
    labels = np.asarray(labels)
    assert len(labels) == len(df_sample)
    d = df_sample[["post_id", "lang"]].assign(cluster=labels)

    out = {
        "n_clusters": int(len(set(labels) - {-1})),
        "noise_share": float((labels == -1).mean()),
    }

    g = gold[(gold["split"] == split) & ~gold["topic"].isin(exclude_topics)]
    m = d.merge(g[["post_id", "topic"]], on="post_id")
    out["gold_coverage"] = float((m["cluster"] != -1).mean())
    m = m[m["cluster"] != -1]

    h, c, v = homogeneity_completeness_v_measure(m["topic"], m["cluster"])
    out.update(
        # чи правда, що два пости в одному кластері [0..1]
        ARI=adjusted_rand_score(m["topic"], m["cluster"]),
        # скільки ми дізнаємось про тему поста, знаючи його кластер [0..1] наскільки знання кластера зменшує невизначеність щодо теми
        NMI=normalized_mutual_info_score(m["topic"], m["cluster"]),
        # like precision in classification
        # дивимось на кожен кластер і питаємо: чи не змішані в ньому різні тем
        homogeneity=h,
        # like recall
        # дивимось на кожну тему і питаємо: чи не розкидана вона по різних кластерах
        completeness=c,
        # like f1-score
        v_measure=v,
        lang_entropy=language_entropy(labels, d["lang"].values),
    )
    return out


def evaluate_both(labels, df_sample, gold, split="dev"):
    """Same labels evaluated on all gold topics and on topical ones only."""
    res_all = evaluate(labels, df_sample, gold, split)
    res_topical = evaluate(labels, df_sample, gold, split, exclude_topics=NON_TOPICAL)
    return pd.DataFrame({"all": res_all, "topical_only": res_topical})



In [ ]:

# Sanity check 1: random labels -> metrics should be close to 0
rng = np.random.default_rng(0)
random_labels = rng.integers(0, 50, size=len(df_sample))
print("Random labels (expect ARI/NMI ~ 0):")
print(evaluate_both(random_labels, df_sample, gold).round(3))

# Sanity check 2: gold topics used as clusters for gold posts -> metrics should be ~1.
# Posts outside gold get a unique dummy cluster so every post has a label.
topic_map = gold.set_index("post_id")["topic"]
codes = {t: i for i, t in enumerate(sorted(gold["topic"].unique()))}
oracle = np.array(
    [codes[topic_map[pid]] if pid in topic_map.index else 1000 + i
     for i, pid in enumerate(df_sample["post_id"])]
)
print("\nOracle labels (expect ARI/NMI/V ~ 1):")
print(evaluate_both(oracle, df_sample, gold).round(3))

## Experiment log
Fix the global seed and define `log_run()`, which appends every evaluated run to `results/metrics.csv`. The random-label run is logged as the floor reference for all later comparisons.

In [4]:
import json
from datetime import datetime
from pathlib import Path

SEED = 42
RESULTS_PATH.parent.mkdir(exist_ok=True)


def log_run(name, params, metrics_df):
    """Append one row per label subset ('all', 'topical_only') to the results log."""
    rows = [
        {
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "run": name,
            "subset": subset,
            "params": json.dumps(params, sort_keys=True),
            **metrics_df[subset].to_dict(),
        }
        for subset in metrics_df.columns
    ]
    new = pd.DataFrame(rows)
    # Re-read and concat so that adding new metric columns later does not break the file
    old = pd.read_csv(RESULTS_PATH) if RESULTS_PATH.exists() else pd.DataFrame()
    pd.concat([old, new], ignore_index=True).to_csv(RESULTS_PATH, index=False)
    return new


In [ ]:


# Floor reference: random labels (random_labels comes from the sanity-check cell)
log_run("random_labels", {"k": 50, "seed": 0}, evaluate_both(random_labels, df_sample, gold))
pd.read_csv(RESULTS_PATH)

## UMAP reduction (MiniLM)
Reduce the 384-d MiniLM embeddings to 15 dimensions with UMAP (cosine metric, `min_dist=0.0`, fixed seed). The result is cached to disk, so it is computed only once.

In [25]:
import umap
from pathlib import Path

ARTIFACTS.mkdir(parents=True, exist_ok=True)

UMAP_PARAMS = {"n_components": 15, "n_neighbors": 15, "min_dist": 0.0, "metric": "cosine"}
umap_path = ARTIFACTS / "minilm_umap15_nn15.npy"

if umap_path.exists():
    # Reuse the cached result instead of recomputing
    X_umap = np.load(umap_path)
else:
    # random_state makes the result reproducible, but disables multi-threading (slower)
    reducer = umap.UMAP(**UMAP_PARAMS, random_state=SEED)
    X_umap = reducer.fit_transform(embeddings)
    np.save(umap_path, X_umap)

assert X_umap.shape == (len(df_sample), UMAP_PARAMS["n_components"])
print("UMAP output:", X_umap.shape)

/home/roman/Documents/university/thesis/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP output: (100000, 15)


## KMeans sweep over k and seeds (MiniLM + UMAP)
Run KMeans for every k with 5 seeds, evaluate each clustering against gold (dev split) and log every run. Report mean ± std of the gold metrics and the run-to-run agreement (pairwise ARI between seeds), so that real differences between k values can be told apart from noise.

In [28]:
import itertools
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

EXPERIMENT = "minilm_umap15_kmeans"
K_GRID = [20, 30, 40, 50, 60, 80]
SEEDS = [SEED + i for i in range(5)]

# Remove rows of this experiment from the log (e.g. from the earlier single-seed sweep),
# so that re-running the cell does not create duplicates
log = pd.read_csv(RESULTS_PATH)
log[log["run"] != EXPERIMENT].to_csv(RESULTS_PATH, index=False)

labels_by_k_seed = {}   # (k, seed) -> cluster labels, kept for later analysis
rows = []

for k in K_GRID:
    per_run = []
    for s in SEEDS:
        # n_init=10: 10 random starts per run, the best one is kept
        labels = KMeans(n_clusters=k, n_init=10, random_state=s).fit_predict(X_umap)
        labels_by_k_seed[(k, s)] = labels

        res = evaluate_both(labels, df_sample, gold)
        log_run(EXPERIMENT, {"repr": "minilm", "umap": UMAP_PARAMS, "k": k, "seed": s}, res)
        per_run.append(res["topical_only"])
    per_run = pd.DataFrame(per_run)

    # Agreement between clusterings produced with different seeds (1 = identical partitions)
    agreement = [
        adjusted_rand_score(labels_by_k_seed[(k, a)], labels_by_k_seed[(k, b)])
        for a, b in itertools.combinations(SEEDS, 2)
    ]

    rows.append({
        "k": k,
        "NMI_mean": per_run["NMI"].mean(), "NMI_std": per_run["NMI"].std(),
        "ARI_mean": per_run["ARI"].mean(), "ARI_std": per_run["ARI"].std(),
        "homog_mean": per_run["homogeneity"].mean(),
        "compl_mean": per_run["completeness"].mean(),
        "lang_entropy": per_run["lang_entropy"].mean(),
        "run_to_run_ARI": float(np.mean(agreement)),
    })

# Compact comparison (topical_only subset); all individual runs are in results/metrics.csv
pd.DataFrame(rows).set_index("k").round(3)

,NMI_mean,NMI_std,ARI_mean,ARI_std,homog_mean,compl_mean,lang_entropy,run_to_run_ARI
k,,,,,,,,
20,0.258,0.003,0.141,0.003,0.271,0.247,0.566,0.784
30,0.278,0.002,0.148,0.002,0.316,0.249,0.565,0.820
40,0.288,0.007,0.148,0.013,0.340,0.250,0.564,0.796
50,0.293,0.003,0.126,0.004,0.359,0.247,0.565,0.800
60,0.302,0.002,0.122,0.003,0.380,0.251,0.567,0.773
80,0.313,0.002,0.099,0.004,0.413,0.252,0.571,0.780


## Supervised reference (upper bound)
Train a logistic regression on the MiniLM embeddings of the gold dev posts with 5-fold cross-validation. This is not a proposed method: it shows how well the topics can be recovered from these embeddings at all, and gives a reference for the unsupervised NMI.

In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, normalized_mutual_info_score

# Row position of every post in df_sample / embeddings
row_of = pd.Series(np.arange(len(df_sample)), index=df_sample["post_id"])

g = gold[(gold["split"] == "dev") & ~gold["topic"].isin(NON_TOPICAL)]
Xg = embeddings[row_of.loc[g["post_id"]].to_numpy()]
yg = g["topic"].to_numpy(dtype=object)

clf = LogisticRegression(max_iter=1000)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
pred = cross_val_predict(clf, Xg, yg, cv=cv)

majority = pd.Series(yg).value_counts(normalize=True).iloc[0]
print(f"posts: {len(yg)} | topics: {len(set(yg))}")
print(f"majority-class accuracy: {majority:.3f}")
print(f"accuracy:   {accuracy_score(yg, pred):.3f}")
print(f"macro F1:   {f1_score(yg, pred, average='macro'):.3f}")
print(f"NMI(true, predicted): {normalized_mutual_info_score(yg, pred):.3f}")

posts: 1674 | topics: 14
majority-class accuracy: 0.174
accuracy:   0.536
macro F1:   0.522
NMI(true, predicted): 0.316


## Grid configuration and helpers
Parameter grids for UMAP, KMeans and HDBSCAN, plus helpers: a disk cache for the UMAP projections, a check whether a run is already in the log (so reruns are resumable), and one function that evaluates and logs a clustering.

In [3]:
import itertools
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import umap
from sklearn.cluster import KMeans, HDBSCAN

# ---------------------------------------------------------------- configuration
REPR = "minilm"
UMAP_GRID = {"n_components": [5, 15, 30], "n_neighbors": [15, 30, 50]}
UMAP_FIXED = {"min_dist": 0.0, "metric": "cosine"}

KMEANS_K = [20, 30, 40, 50, 60, 80]                        # compared at the same k across UMAP configs
HDBSCAN_MIN_CLUSTER_SIZE = [50, 100, 200, 500]
HDBSCAN_MIN_SAMPLES = 10                      # fixed, only min_cluster_size is swept

EXP_KMEANS = "minilm_umap_kmeans"
EXP_HDBSCAN = "minilm_umap_hdbscan"

ARTIFACTS.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------- helpers
def get_umap(embeddings, n_components, n_neighbors):
    """Return the UMAP projection, computed once and cached on disk."""
    path = ARTIFACTS / f"{REPR}_umap{n_components}_nn{n_neighbors}.npy"
    if path.exists():
        return np.load(path)
    t0 = time.time()
    # random_state makes the result reproducible but disables multi-threading
    reducer = umap.UMAP(
        n_components=n_components, n_neighbors=n_neighbors, **UMAP_FIXED, random_state=SEED
    )
    X_red = reducer.fit_transform(embeddings)
    np.save(path, X_red)
    print(f"  UMAP n_components={n_components}, n_neighbors={n_neighbors}: {time.time() - t0:.0f}s")
    return X_red


def is_logged(experiment, params):
    """True if a run with exactly these parameters is already in the results log."""
    if not RESULTS_PATH.exists():
        return False
    log = pd.read_csv(RESULTS_PATH, usecols=["run", "params"])
    key = json.dumps(params, sort_keys=True)
    return bool(((log["run"] == experiment) & (log["params"] == key)).any())


def evaluate_and_log(experiment, params, labels):
    """Evaluate a clustering on both gold subsets and append it to the log."""
    if len(set(labels) - {-1}) == 0:
        print(f"  skipped (every post is noise): {params}")
        return
    log_run(experiment, params, evaluate_both(labels, df_sample, gold))

/home/roman/Documents/university/thesis/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## UMAP grid with KMeans
For every (n_components, n_neighbors) pair: compute or load the UMAP projection, then run KMeans at each k in the grid. Projections are cached in `artifacts/embeddings`, runs already in the log are skipped.

In [32]:
umap_configs = list(itertools.product(UMAP_GRID["n_components"], UMAP_GRID["n_neighbors"]))

for i, (nc, nn) in enumerate(umap_configs, 1):
    print(f"[{i}/{len(umap_configs)}] n_components={nc}, n_neighbors={nn}")
    X_red = get_umap(embeddings, nc, nn)

    for k in KMEANS_K:
        params = {"repr": REPR, "n_components": nc, "n_neighbors": nn, **UMAP_FIXED, "k": k, "seed": SEED}
        if is_logged(EXP_KMEANS, params):
            continue
        # n_init=10: ten random starts, the best one is kept
        labels = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(X_red)
        evaluate_and_log(EXP_KMEANS, params, labels)

print("done")

[1/9] n_components=5, n_neighbors=15
  UMAP n_components=5, n_neighbors=15: 88s
[2/9] n_components=5, n_neighbors=30
  UMAP n_components=5, n_neighbors=30: 117s
[3/9] n_components=5, n_neighbors=50
  UMAP n_components=5, n_neighbors=50: 159s
[4/9] n_components=15, n_neighbors=15
[5/9] n_components=15, n_neighbors=30
  UMAP n_components=15, n_neighbors=30: 150s
[6/9] n_components=15, n_neighbors=50
  UMAP n_components=15, n_neighbors=50: 213s
[7/9] n_components=30, n_neighbors=15
  UMAP n_components=30, n_neighbors=15: 185s
[8/9] n_components=30, n_neighbors=30
  UMAP n_components=30, n_neighbors=30: 229s
[9/9] n_components=30, n_neighbors=50
  UMAP n_components=30, n_neighbors=50: 291s
done


## HDBSCAN sweep on the same projections
HDBSCAN needs no k: it finds dense regions and labels everything else as noise (-1). For every cached UMAP projection, sweep `min_cluster_size`. Noise share and gold coverage are logged next to the gold metrics, because HDBSCAN scores only the posts it did not discard.

In [34]:
for i, (nc, nn) in enumerate(umap_configs, 1):
    print(f"[{i}/{len(umap_configs)}] n_components={nc}, n_neighbors={nn}")
    X_red = get_umap(embeddings, nc, nn)   # loaded from cache

    for mcs in HDBSCAN_MIN_CLUSTER_SIZE:
        params = {
            "repr": REPR, "n_components": nc, "n_neighbors": nn, **UMAP_FIXED,
            "min_cluster_size": mcs, "min_samples": HDBSCAN_MIN_SAMPLES, "seed": SEED,
        }
        if is_logged(EXP_HDBSCAN, params):
            continue
        t0 = time.time()
        labels = HDBSCAN(
            min_cluster_size=mcs, min_samples=HDBSCAN_MIN_SAMPLES, n_jobs=-1
        ).fit_predict(X_red)
        evaluate_and_log(EXP_HDBSCAN, params, labels)
        print(f"  min_cluster_size={mcs}: {len(set(labels) - {-1})} clusters, "
              f"noise {(labels == -1).mean():.1%}, {time.time() - t0:.0f}s")

print("done")

[1/9] n_components=5, n_neighbors=15
[2/9] n_components=5, n_neighbors=30
[3/9] n_components=5, n_neighbors=50
[4/9] n_components=15, n_neighbors=15
[5/9] n_components=15, n_neighbors=30
[6/9] n_components=15, n_neighbors=50
[7/9] n_components=30, n_neighbors=15
[8/9] n_components=30, n_neighbors=30
  min_cluster_size=50: 190 clusters, noise 51.4%, 205s
  min_cluster_size=100: 106 clusters, noise 49.7%, 207s
  min_cluster_size=200: 59 clusters, noise 46.7%, 206s
  min_cluster_size=500: 28 clusters, noise 45.1%, 206s
[9/9] n_components=30, n_neighbors=50
  min_cluster_size=50: 176 clusters, noise 55.0%, 201s
  min_cluster_size=100: 96 clusters, noise 50.8%, 198s
  min_cluster_size=200: 52 clusters, noise 50.8%, 204s
  min_cluster_size=500: 30 clusters, noise 49.9%, 204s
done


## Grid results
Read the runs back from `results/metrics.csv` and show them as grids (rows: n_components, columns: n_neighbors) on the topical-only subset. KMeans is compared at fixed k. For HDBSCAN, read the gold metrics together with the noise share.

In [35]:
from IPython.display import display


def load_results(experiment, subset="topical_only"):
    """Runs of one experiment as a flat table: metrics + one column per parameter."""
    log = pd.read_csv(RESULTS_PATH)
    log = log[(log["run"] == experiment) & (log["subset"] == subset)].reset_index(drop=True)
    params = pd.json_normalize(log["params"].apply(json.loads).tolist())
    out = pd.concat([log.drop(columns="params"), params], axis=1)
    return out.drop_duplicates(subset=list(params.columns), keep="last")


def show_grid(df, value, title):
    grid = df.pivot(index="n_components", columns="n_neighbors", values=value)
    print(title)
    display(grid.style.format("{:.3f}").background_gradient(axis=None))


# KMeans: rows are UMAP configs, columns are k
km = load_results(EXP_KMEANS)
for metric in ["NMI", "ARI", "lang_entropy"]:
    table = km.pivot_table(index=["n_components", "n_neighbors"], columns="k", values=metric)
    print(f"KMeans: {metric}")
    display(table.style.format("{:.3f}").background_gradient(axis=None))

# HDBSCAN: one grid per min_cluster_size; gold metrics together with noise
hd = load_results(EXP_HDBSCAN)
for mcs, g in hd.groupby("min_cluster_size"):
    for metric in ["NMI", "noise_share", "n_clusters"]:
        show_grid(g, metric, f"HDBSCAN min_cluster_size={mcs}: {metric}")

KMeans: NMI


KMeans: ARI


KMeans: lang_entropy


HDBSCAN min_cluster_size=50: NMI


n_neighbors,15,30,50
n_components,,,
5,0.459,0.480,0.490
15,0.461,0.480,0.480
30,0.466,0.477,0.479


HDBSCAN min_cluster_size=50: noise_share


... (17 outputs left out)

HDBSCAN min_cluster_size=500: noise_share


n_neighbors,15,30,50
n_components,,,
5,0.475,0.502,0.501
15,0.456,0.483,0.472
30,0.482,0.451,0.499


HDBSCAN min_cluster_size=500: n_clusters


n_neighbors,15,30,50
n_components,,,
5,36.000,31.000,30.000
15,34.000,29.000,28.000
30,34.000,28.000,30.000


## Summary: UMAP and clustering grid on MiniLM embeddings

**What we did.** We reduced the 384-d MiniLM embeddings with UMAP (3 values of `n_components` × 3 values of `n_neighbors`, 9 configurations) and clustered each projection in two ways. KMeans was run for k = 20, 30, 40, 50, 60, 80. HDBSCAN was run for `min_cluster_size` = 50, 100, 200, 500. Every run was scored against the gold dev set (topical posts only) and logged to `results/metrics.csv`. The test split was not used.

**Reference points.**
- Random cluster labels score NMI ≈ 0.06. This is the floor.
- A supervised logistic regression on the same embeddings (5-fold CV) reaches 54% accuracy and macro F1 0.52, so the gold topics are only moderately separable even with labels.

### Findings

**1. UMAP settings barely matter.**
At k = 50, NMI is 0.292–0.302 and ARI is 0.114–0.129 across all 9 configurations. The difference is about 0.01, which is close to the noise between seeds (NMI std ≤ 0.007).
- `n_components` (5, 15, 30) has no visible effect.
- `n_neighbors = 50` is slightly better than 15 or 30 (+0.005 to +0.02 NMI, mostly at small k).
- (`n_components` = 5, `n_neighbors` = 50) is among the best for almost every k and is also the cheapest to compute.

**2. KMeans: no single best k.**
- NMI and homogeneity keep growing with k (NMI 0.25 → 0.31 from k = 20 to 80), because smaller clusters are purer.
- ARI is best at k = 20–40 (≈ 0.15) and drops to ≈ 0.09 at k = 80.
- Completeness stays at ≈ 0.25 for every k: topics are split across several clusters regardless of k.
- The gold set has only 14 topical labels, so k = 30–50 is a reasonable range.
- Different seeds agree on about 80% of the partition (run-to-run ARI 0.77–0.82).

**3. Clusters group by topic, not by language.**
Language entropy is 0.54–0.58 for all methods, close to the 0.61 of random labels. Posts in different languages are mixed inside clusters, as we wanted from a multilingual encoder.

**4. HDBSCAN looks better, but the comparison is not fair yet.**
HDBSCAN reaches NMI 0.38–0.49 against 0.25–0.32 for KMeans. However:
- It marks 45–56% of all posts as noise, in every configuration. `min_cluster_size` only changes the number of clusters (28 to 234), not the share of noise.
- Metrics are computed only on the posts it keeps, so only about half of the gold posts are scored.
- Within `min_cluster_size` = 50, NMI tends to be higher where more posts are discarded.
- NMI 0.49 is higher than the supervised reference (0.32). This suggests that the kept posts are simply easier, and the clustering itself is not necessarily better. (NMI also depends on the number of clusters, so these numbers are only a rough comparison.)
- Noise is not just the "chatty" half. The share of noise is almost the same for topical and non-topical gold posts.

There is a hint of a real advantage: at `min_cluster_size` = 500 HDBSCAN has about 30 clusters, like KMeans with k = 30, and still gets NMI ≈ 0.40 and ARI ≈ 0.26 (KMeans k = 30: 0.28 and 0.15). But this is measured on posts that HDBSCAN selected itself.

### Conclusion
- The KMeans baseline on MiniLM + UMAP is stable and clearly above the random floor (NMI ≈ 0.29 vs 0.06), but not very high.
- UMAP hyperparameters are not worth more tuning in this range.
- The open question is a fair KMeans vs HDBSCAN comparison.

### Caveats
- One UMAP seed per configuration; seed variance was measured only for KMeans.
- Gold labels come from an LLM and were not manually verified.
- Evaluation uses the dev split only (about 1,700 topical posts), so small differences should not be over-interpreted.

### Next steps
1. Evaluate KMeans on the same posts that HDBSCAN keeps, so both are scored on identical data.
2. Add an HDBSCAN variant that assigns noise posts to the nearest cluster, so both methods are scored on all gold posts.
3. Fix one UMAP configuration (for example `n_components` = 5, `n_neighbors` = 50) and use it to compare the other representations (TF-IDF, fastText, a stronger encoder).

## Fair KMeans vs HDBSCAN
HDBSCAN discards about half of the posts and is scored only on the rest, so its NMI is not comparable with KMeans. We compare them in two ways on the fixed projection (UMAP 5D, n_neighbors=50). First, KMeans with the same number of clusters is scored on exactly the posts HDBSCAN keeps. Second, HDBSCAN noise posts are assigned to a cluster via their nearest clustered neighbours, so both methods are scored on all gold posts.

# RUN THIS NEXT

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

UMAP_NC, UMAP_NN = 5, 50
X_red = get_umap(embeddings, UMAP_NC, UMAP_NN)   # loaded from cache
BASE = {"repr": REPR, "n_components": UMAP_NC, "n_neighbors": UMAP_NN, **UMAP_FIXED}

EXP_KM_KEPT = "fair_kmeans_on_hdbscan_kept"   # KMeans scored only on posts HDBSCAN keeps
EXP_KM_ALL = "fair_kmeans_all"                # KMeans scored on all gold posts
EXP_HDB_FILLED = "hdbscan_noise_assigned"     # HDBSCAN with noise reassigned, all gold posts
KM_SEEDS = [SEED + i for i in range(3)]
KNN_K = 15

# Drop earlier rows of these experiments so that re-running the cell does not duplicate them
log = pd.read_csv(RESULTS_PATH)
log[~log["run"].isin([EXP_KM_KEPT, EXP_KM_ALL, EXP_HDB_FILLED])].to_csv(RESULTS_PATH, index=False)


def run_and_log(experiment, params, labels):
    """Evaluate on both gold subsets, log, and return the topical-only metrics."""
    res = evaluate_both(labels, df_sample, gold)
    log_run(experiment, params, res)
    return res["topical_only"]


rows = []
for mcs in HDBSCAN_MIN_CLUSTER_SIZE:
    hdb = HDBSCAN(min_cluster_size=mcs, min_samples=HDBSCAN_MIN_SAMPLES, n_jobs=-1).fit_predict(X_red)
    kept = hdb != -1
    k = len(set(hdb) - {-1})
    hdb_params = {**BASE, "min_cluster_size": mcs, "min_samples": HDBSCAN_MIN_SAMPLES}

    # HDBSCAN as before: noise excluded from the metrics (already in the log under EXP_HDBSCAN)
    r_hdb = evaluate_both(hdb, df_sample, gold)["topical_only"]

    # HDBSCAN + noise assignment: each noise post gets the majority cluster of its 15 nearest clustered posts
    knn = KNeighborsClassifier(n_neighbors=KNN_K).fit(X_red[kept], hdb[kept])
    hdb_filled = hdb.copy()
    hdb_filled[~kept] = knn.predict(X_red[~kept])
    r_fill = run_and_log(EXP_HDB_FILLED, {**hdb_params, "knn": KNN_K}, hdb_filled)

    # KMeans with the same number of clusters as HDBSCAN found, averaged over seeds
    km_kept, km_all = [], []
    for s in KM_SEEDS:
        km = KMeans(n_clusters=k, n_init=10, random_state=s).fit_predict(X_red)
        p = {**BASE, "k": k, "seed": s, "matched_to_mcs": mcs}
        km_all.append(run_and_log(EXP_KM_ALL, p, km))
        # Labels outside the HDBSCAN-kept set become -1, so evaluate() skips them like HDBSCAN noise
        km_kept.append(run_and_log(EXP_KM_KEPT, p, np.where(kept, km, -1)))
    km_kept = pd.DataFrame(km_kept).mean()
    km_all = pd.DataFrame(km_all).mean()

    rows.append({
        "min_cluster_size": mcs, "k": k, "kept_share": kept.mean(),
        "HDB_kept_NMI": r_hdb["NMI"], "KM_kept_NMI": km_kept["NMI"],
        "HDB_kept_ARI": r_hdb["ARI"], "KM_kept_ARI": km_kept["ARI"],
        "HDB_filled_NMI": r_fill["NMI"], "KM_all_NMI": km_all["NMI"],
        "HDB_filled_ARI": r_fill["ARI"], "KM_all_ARI": km_all["ARI"],
    })
    print(f"min_cluster_size={mcs}: k={k}, kept {kept.mean():.1%}")

fair = pd.DataFrame(rows).set_index("min_cluster_size")
display(fair.style.format("{:.3f}"))

/home/roman/Documents/university/thesis/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
